In [ ]:
##data ingestion 

import os 


os.environ["GOOGLE_API_KEY"] = "your_api_key" 

In [ ]:
from google import genai

gemini_client = genai.Client(api_key="your_api_key")

def get_embedding(text):
    response = gemini_client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=text
    )
    return response.embeddings[0].values

In [30]:
embed =get_embedding("RAG TECHNOLOGY")
print(len(embed))

3072


In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF
loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

# Split the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(data)

In [32]:
documents

[Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue'),
 Document(metadata={'producer': 'West Corporation using ABCpdf', 'creator': 'PyPDF', 'creationdate': '2024-05-30T20:06:12+00:00', 'title': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results', 'source': 'https://investors.mongodb.com/node/12236/pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='NEW YORK, May 30, 2

In [33]:
## prepare documents for insertion 
docs_to_insert = [{
    "text": doc.page_content,
    "embedding": get_embedding(doc.page_content)
} for doc in documents]

In [34]:
docs_to_insert

[{'text': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue',
  'embedding': [-0.027504634,
   0.01827896,
   -0.00640047,
   -0.0922297,
   -0.0034582585,
   0.027722841,
   0.021760482,
   -0.012786122,
   -0.031444773,
   0.04272564,
   0.00446207,
   -0.005696556,
   0.012480129,
   -0.014734025,
   0.11720198,
   -0.0012443359,
   0.013407363,
   -0.016932404,
   -0.0009673481,
   -0.013949263,
   -0.024083724,
   0.003230869,
   0.0008372245,
   -0.010232543,
   -0.011773399,
   -0.004680616,
   0.0047065765,
   0.001529042,
   0.030352505,
   -0.0047439868,
   -0.004748192,
   -0.03420273,
   0.02168162,
   0.0009172202,
   -0.00041894676,
   0.013797669,
   0.018960517,
   0.0017327509,
   0.0063527357,
   0.0101381

In [ ]:
from pymongo import MongoClient

# Connect to your MongoDB deployment
client = MongoClient("")
collection =  client["rag_db"]["test"]

# Insert documents into the collection
result = collection.insert_many(docs_to_insert)
result

InsertManyResult([ObjectId('69b138119d2e20e7e1396592'), ObjectId('69b138119d2e20e7e1396593'), ObjectId('69b138119d2e20e7e1396594'), ObjectId('69b138119d2e20e7e1396595'), ObjectId('69b138119d2e20e7e1396596'), ObjectId('69b138119d2e20e7e1396597'), ObjectId('69b138119d2e20e7e1396598'), ObjectId('69b138119d2e20e7e1396599'), ObjectId('69b138119d2e20e7e139659a'), ObjectId('69b138119d2e20e7e139659b'), ObjectId('69b138119d2e20e7e139659c'), ObjectId('69b138119d2e20e7e139659d'), ObjectId('69b138119d2e20e7e139659e'), ObjectId('69b138119d2e20e7e139659f'), ObjectId('69b138119d2e20e7e13965a0'), ObjectId('69b138119d2e20e7e13965a1'), ObjectId('69b138119d2e20e7e13965a2'), ObjectId('69b138119d2e20e7e13965a3'), ObjectId('69b138119d2e20e7e13965a4'), ObjectId('69b138119d2e20e7e13965a5'), ObjectId('69b138119d2e20e7e13965a6'), ObjectId('69b138119d2e20e7e13965a7'), ObjectId('69b138119d2e20e7e13965a8'), ObjectId('69b138119d2e20e7e13965a9'), ObjectId('69b138119d2e20e7e13965aa'), ObjectId('69b138119d2e20e7e13965

In [36]:
doc = collection.find_one()
print(doc)

{'_id': ObjectId('69b058e396a37715fae622a2'), 'text': 'MongoDB, Inc. Announces First Quarter Fiscal 2025 Financial Results\nMay 30, 2024\nFirst Quarter Fiscal 2025 Total Revenue of $450.6 million, up 22% Year-over-Year\nContinued Strong Customer Growth with Over 49,200 Customers as of April 30, 2024\nMongoDB Atlas Revenue up 32% Year-over-Year; 70% of Total Q1 Revenue', 'embedding': [-0.027441317, 0.017902756, -0.0059731603, -0.092204735, -0.0036032198, 0.028047504, 0.021885656, -0.012789561, -0.03116559, 0.04267548, 0.0047116666, -0.0055625117, 0.011865189, -0.015367229, 0.11711647, -0.0014353594, 0.013923433, -0.016810458, -0.0013405106, -0.014109084, -0.024343813, 0.0039538727, 0.0006993569, -0.009993126, -0.011056254, -0.0043809903, 0.0042980816, 0.001289262, 0.029905839, -0.0051569478, -0.004966491, -0.034612007, 0.021087386, 0.0012910061, -0.00081970845, 0.013060603, 0.01952785, 0.0015043471, 0.00663434, 0.010543769, -0.013681158, 0.014147875, -0.0024118952, -0.019959858, -0.0018

In [37]:
from pymongo.operations import SearchIndexModel
import time

# Create your index model, then create the search index
index_name="vector_index"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 3072,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)
collection.create_search_index(model=search_index_model)

# Wait for initial sync to complete
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


In [51]:
query_embedding = get_embedding("AI Technology")
collection.ragpdf.aggregate([
  {
    "$vectorSearch": {
      "index": "vector_index",
      "path": "embedding",
      "queryVector":query_embedding,
      "numCandidates":3072 ,
      "limit": 5
    }
  }
])

In [56]:
# Define a function to run vector search queries
def get_query_results(query):
  """Gets results from a vector search query."""

  query_embedding = get_embedding(query)
  print(query_embedding)
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index",
              "queryVector": query_embedding,
              "path": "embedding",
              "numCandidates":3072,
              "limit": 10
            }
      }, {
            "$project": {
              "_id": 0,
              "text": 1
         }
      }
  ]

  results = collection.aggregate(pipeline)
  print(results)

  array_of_results = []
  for doc in results:
      array_of_results.append(doc)
  return array_of_results



In [57]:
# Test the function with a sample query
get_query_results("mongodb vector search")

[-0.014239147, -0.024174843, -0.021698033, -0.08782263, -0.0063040694, 0.01770771, -0.0039573903, 0.015209142, 0.010649297, 0.011595481, -0.013331073, 0.0071023908, 0.025141504, -0.0029159286, 0.117226124, -0.017823858, -0.011269154, 0.0012105452, -0.014692765, -0.015876913, -0.014701855, -0.01218378, 0.023306351, -0.0137956785, 0.004935111, -0.0068785734, 0.022452341, -0.0011617274, 0.011430654, 0.011640916, 0.0045052776, 0.005592334, 0.03627956, 0.016016783, -0.008365317, 0.009051818, 0.01548429, 0.013235006, 0.011810955, 0.012832296, 0.010700637, -0.02031234, -0.006503932, -0.012752467, 0.006070917, 0.025126025, 0.025296358, -0.021507813, 0.009536326, 0.017489277, 0.0132235745, -0.017130068, 0.00039365145, -0.14940184, 0.02198867, 0.0204126, 0.0038259018, -0.0060910513, 0.018686997, -0.007509384, -0.008047909, 0.01875273, -0.015613037, 0.002283354, 0.0042430563, 0.008059505, 0.02710848, -0.0012380818, -0.009078494, -0.015815796, -0.0030079796, -0.02227848, 0.0046562525, -0.045333248

[{'text': "About MongoDB\nHeadquartered in New York, MongoDB's mission is to empower innovators to create, transform, and disrupt industries by unleashing the power of\nsoftware and data. Built by developers, for developers, MongoDB's developer data platform is a database with an integrated set of related services"},
 {'text': "About MongoDB\nHeadquartered in New York, MongoDB's mission is to empower innovators to create, transform, and disrupt industries by unleashing the power of\nsoftware and data. Built by developers, for developers, MongoDB's developer data platform is a database with an integrated set of related services"},
 {'text': "About MongoDB\nHeadquartered in New York, MongoDB's mission is to empower innovators to create, transform, and disrupt industries by unleashing the power of\nsoftware and data. Built by developers, for developers, MongoDB's developer data platform is a database with an integrated set of related services"},
 {'text': "About MongoDB\nHeadquartered in 

In [ ]:
from google import genai

gemini_client = genai.Client(api_key="your_api_key")

query = "Features of MongoDB ?"

context_docs = get_query_results(query)
context_string = "\n\n".join([doc["text"] for doc in context_docs])

prompt = f"""

You are an AI assistant.

Use the context below to answer the question clearly and in detail.


Context:
{context_string}

Question:
{query}

Instructions:
- Combine information from all context chunks
- Avoid repeating the same sentence
- Write a clear explanation
- Summarize the information

"""

response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

[-0.017680638, -0.03433029, -0.01034353, -0.03661072, 0.007214379, 0.012685318, 0.012674886, 0.012922611, 0.023560148, 0.026072666, -0.017175052, -0.01116184, 0.03038241, 0.024642954, 0.11520306, 0.005985958, -0.0040339963, -0.017129587, -0.024372134, 0.01149223, -0.006305356, -0.018543761, -0.0012766218, -0.0013799998, -0.025989283, 0.0038852212, 0.010650678, -0.028309014, 0.018428335, 0.029984098, 0.018320281, 0.013644766, 0.019461876, 0.010157642, -0.026037037, 0.009524546, 0.034722414, 0.031310383, 0.01385575, -0.0069682607, -0.012510907, 0.016298436, 0.0007496951, 0.0021690174, 0.009790642, 0.021735752, 0.0081241075, 0.004478457, -0.012915865, 0.013510164, 0.024617478, 0.007818409, 0.0074334573, -0.17205597, 0.017744632, -0.0061209127, -0.008112197, -0.0102298735, 0.017723618, -0.004954235, -0.011992273, 0.0074263075, -0.00679021, -0.0024076805, 0.018906163, -0.015060959, 0.015735306, 0.018453019, -0.010048648, -0.016308326, -0.010596449, -0.010046595, -0.00045943112, -0.02886951,

In [47]:
models = gemini_client.models.list()

for m in models:
    print(m.name, m.display_name)

models/gemini-2.5-flash Gemini 2.5 Flash
models/gemini-2.5-pro Gemini 2.5 Pro
models/gemini-2.0-flash Gemini 2.0 Flash
models/gemini-2.0-flash-001 Gemini 2.0 Flash 001
models/gemini-2.0-flash-exp-image-generation Gemini 2.0 Flash (Image Generation) Experimental
models/gemini-2.0-flash-lite-001 Gemini 2.0 Flash-Lite 001
models/gemini-2.0-flash-lite Gemini 2.0 Flash-Lite
models/gemini-2.5-flash-preview-tts Gemini 2.5 Flash Preview TTS
models/gemini-2.5-pro-preview-tts Gemini 2.5 Pro Preview TTS
models/gemma-3-1b-it Gemma 3 1B
models/gemma-3-4b-it Gemma 3 4B
models/gemma-3-12b-it Gemma 3 12B
models/gemma-3-27b-it Gemma 3 27B
models/gemma-3n-e4b-it Gemma 3n E4B
models/gemma-3n-e2b-it Gemma 3n E2B
models/gemini-flash-latest Gemini Flash Latest
models/gemini-flash-lite-latest Gemini Flash-Lite Latest
models/gemini-pro-latest Gemini Pro Latest
models/gemini-2.5-flash-lite Gemini 2.5 Flash-Lite
models/gemini-2.5-flash-image Nano Banana
models/gemini-2.5-flash-lite-preview-09-2025 Gemini 2.5 Fl

Your code implements a complete **Retrieval-Augmented Generation (RAG) pipeline** using MongoDB and Google’s Gemini model. First, it takes raw documents, generates embeddings for each one, and stores both the text and its embedding in a MongoDB collection. Then, it creates a **vector search index** on the embedding field, which allows MongoDB to efficiently perform similarity searches using cosine distance. This index is essential because it enables fast and accurate retrieval of the most relevant documents when a query is made.

When a user asks a question, the code converts that query into an embedding and runs a **vector search** against the indexed documents. The top matching results are returned, and their text is combined into a context string. This context is then passed into Gemini along with the original question, forming a structured prompt. Gemini uses the retrieved context to generate a clear, detailed, and summarized answer. In short, your code connects storage (MongoDB), retrieval (vector search), and generation (Gemini) into one workflow, so the AI can answer questions based on the knowledge stored in your database rather than relying only on its built-in training.
